# Student Attrition Prediction Using Data Mining

**Developer:** Sanusi Shafii  
**Institution:** Federal University Dutsin-Ma (FUDMA)  
**Email:** sanusi33@gmail.com  
**Date:** February 2026

---

## Project Overview

Advanced student attrition prediction system with comprehensive feature analysis.

**Target Accuracy:** Minimum 80%

**Key Strategies:**
- Comprehensive feature engineering from all available data
- Advanced ensemble methods with deep hyperparameter tuning
- Multiple balancing techniques
- Automated feature importance analysis

---

In [3]:
!pip install -q xgboost lightgbm catboost scikit-learn imbalanced-learn
!pip install -q plotly seaborn umap-learn shap mlxtend joblib tqdm optuna

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Student_Attrition_Project'
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/visualizations', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/reports', exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Student_Attrition_Project


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import *
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import *
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE, RFECV
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTEENN, SMOTETomek

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from umap import UMAP
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import shap
from scipy import stats
from tqdm.auto import tqdm
import joblib
from datetime import datetime
import json
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pio.renderers.default = 'colab'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully")

Libraries imported successfully


## Data Loading

Provide your dataset URL below.

In [6]:
# Load dataset
dataset_url = "/content/dataset.csv"

df = pd.read_csv(dataset_url)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nAll available columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

display(df.head())
display(df.info())

Dataset loaded: 4424 rows, 35 columns

All available columns:
 1. Marital status
 2. Application mode
 3. Application order
 4. Course
 5. Daytime/evening attendance
 6. Previous qualification
 7. Nacionality
 8. Mother's qualification
 9. Father's qualification
10. Mother's occupation
11. Father's occupation
12. Displaced
13. Educational special needs
14. Debtor
15. Tuition fees up to date
16. Gender
17. Scholarship holder
18. Age at enrollment
19. International
20. Curricular units 1st sem (credited)
21. Curricular units 1st sem (enrolled)
22. Curricular units 1st sem (evaluations)
23. Curricular units 1st sem (approved)
24. Curricular units 1st sem (grade)
25. Curricular units 1st sem (without evaluations)
26. Curricular units 2nd sem (credited)
27. Curricular units 2nd sem (enrolled)
28. Curricular units 2nd sem (evaluations)
29. Curricular units 2nd sem (approved)
30. Curricular units 2nd sem (grade)
31. Curricular units 2nd sem (without evaluations)
32. Unemployment rate
33. Infl

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,10,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,4,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,10,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,4,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,10,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 35 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Nacionality                                     4424 non-null   int64  
 7   Mother's qualification                          4424 non-null   int64  
 8   Father's qualification                          4424 non-null   int64  
 9   Mother's occupation                      

None

## Comprehensive Data Analysis

In [7]:
print("Dataset Analysis")
print("="*80)

# Statistical summary
print("\nStatistical Summary:")
display(df.describe())

# Missing values
print("\nMissing Values Analysis:")
missing = df.isnull().sum()
if missing.any():
    missing_df = pd.DataFrame({
        'Column': missing[missing > 0].index,
        'Missing': missing[missing > 0].values,
        'Percentage': (missing[missing > 0].values / len(df) * 100).round(2)
    })
    display(missing_df)
else:
    print("No missing values detected")

# Data types
print("\nData Types:")
print(f"Numerical features: {len(df.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"Categorical features: {len(df.select_dtypes(include=['object']).columns)}")

# Target analysis
if 'Target' in df.columns:
    print("\nTarget Variable Analysis:")
    target_counts = df['Target'].value_counts()
    target_pct = (df['Target'].value_counts(normalize=True) * 100).round(2)

    target_summary = pd.DataFrame({
        'Class': target_counts.index,
        'Count': target_counts.values,
        'Percentage': target_pct.values
    })
    display(target_summary)

    # Check imbalance
    imbalance_ratio = target_pct.max() / target_pct.min()
    print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}")
    if imbalance_ratio > 1.5:
        print("Note: Significant class imbalance detected. Will apply advanced balancing.")

Dataset Analysis

Statistical Summary:


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
count,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000
mean,1.178571,6.886980,1.727848,9.899186,0.890823,2.531420,1.254521,12.322107,16.455244,7.317812,7.819168,0.548373,0.011528,0.113698,0.880651,0.351718,0.248418,23.265145,0.024864,0.709991,6.270570,8.299051,4.706600,10.640822,0.137658,0.541817,6.232143,8.063291,4.435805,10.230206,0.150316,11.566139,1.228029,0.001969
std,0.605747,5.298964,1.313793,4.331792,0.311897,3.963707,1.748447,9.026251,11.044800,3.997828,4.856692,0.497711,0.106760,0.317480,0.324235,0.477560,0.432144,7.587816,0.155729,2.360507,2.480178,4.179106,3.094238,4.843663,0.690880,1.918546,2.195951,3.947951,3.014764,5.210808,0.753774,2.663850,1.382711,2.269935
min,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.600000,-0.800000,-4.060000
25%,1.000000,1.000000,1.000000,6.000000,1.000000,1.000000,1.000000,2.000000,3.000000,5.000000,5.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,19.000000,0.000000,0.000000,5.000000,6.000000,3.000000,11.000000,0.000000,0.000000,5.000000,6.000000,2.000000,10.750000,0.000000,9.400000,0.300000,-1.700000
50%,1.000000,8.000000,1.000000,10.000000,1.000000,1.000000,1.000000,13.000000,14.000000,6.000000,8.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,20.000000,0.000000,0.000000,6.000000,8.000000,5.000000,12.285714,0.000000,0.000000,6.000000,8.000000,5.000000,12.200000,0.000000,11.100000,1.400000,0.320000
75%,1.000000,12.000000,2.000000,13.000000,1.000000,1.000000,1.000000,22.000000,27.000000,10.000000,10.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,25.000000,0.000000,0.000000,7.000000,10.000000,6.000000,13.400000,0.000000,0.000000,7.000000,10.000000,6.000000,13.333333,0.000000,13.900000,2.600000,1.790000
max,6.000000,18.000000,9.000000,17.000000,1.000000,17.000000,21.000000,29.000000,34.000000,32.000000,46.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,70.000000,1.000000,20.000000,26.000000,45.000000,26.000000,18.875000,12.000000,19.000000,23.000000,33.000000,20.000000,18.571429,12.000000,16.200000,3.700000,3.510000



Missing Values Analysis:
No missing values detected

Data Types:
Numerical features: 34
Categorical features: 1

Target Variable Analysis:


,Class,Count,Percentage
0,Graduate,2209,49.93
1,Dropout,1421,32.12
2,Enrolled,794,17.95



Class imbalance ratio: 2.78
Note: Significant class imbalance detected. Will apply advanced balancing.


In [8]:
# Correlation with target
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'Target' in df.columns and len(numerical_cols) > 0:
    # Encode target for correlation
    le_temp = LabelEncoder()
    df_temp = df.copy()
    df_temp['Target_Encoded'] = le_temp.fit_transform(df_temp['Target'])

    # Calculate correlation with target
    correlations = df_temp[numerical_cols].corrwith(df_temp['Target_Encoded']).abs().sort_values(ascending=False)

    print("\nTop 20 Features Correlated with Target:")
    print("="*80)
    corr_df = pd.DataFrame({
        'Feature': correlations.head(20).index,
        'Correlation': correlations.head(20).values
    })
    display(corr_df)

    # Visualization
    fig = go.Figure([
        go.Bar(x=corr_df['Correlation'], y=corr_df['Feature'], orientation='h')
    ])
    fig.update_layout(title='Feature Correlation with Target',
                     xaxis_title='Absolute Correlation',
                     yaxis_title='Feature',
                     height=600,
                     yaxis=dict(autorange="reversed"))
    fig.show()
    fig.write_html(f"{PROJECT_DIR}/visualizations/target_correlation.html")


Top 20 Features Correlated with Target:


,Feature,Correlation
0,Curricular units 2nd sem (approved),0.624157
1,Curricular units 2nd sem (grade),0.566827
2,Curricular units 1st sem (approved),0.529123
3,Curricular units 1st sem (grade),0.485207
4,Tuition fees up to date,0.409827
5,Scholarship holder,0.297595
6,Age at enrollment,0.243438
7,Debtor,0.240999
8,Gender,0.229270
9,Application mode,0.212025


## Advanced Preprocessing & Feature Engineering

In [9]:
print("Advanced Preprocessing Pipeline")
print("="*80)

df_processed = df.copy()

# Step 1: Handle missing values
print("\nStep 1: Handling missing values...")
for col in df_processed.columns:
    if df_processed[col].isnull().sum() > 0:
        if df_processed[col].dtype in ['int64', 'float64']:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
        elif col != 'Target':
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

# Step 2: Advanced feature engineering
print("\nStep 2: Creating advanced features...")

numerical_features = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'Target' in numerical_features:
    numerical_features.remove('Target')

initial_features = len(df_processed.columns)

# Create statistical aggregation features
if len(numerical_features) >= 2:
    df_processed['feature_mean'] = df_processed[numerical_features].mean(axis=1)
    df_processed['feature_std'] = df_processed[numerical_features].std(axis=1)
    df_processed['feature_max'] = df_processed[numerical_features].max(axis=1)
    df_processed['feature_min'] = df_processed[numerical_features].min(axis=1)
    df_processed['feature_range'] = df_processed['feature_max'] - df_processed['feature_min']

# Create interaction features for highly correlated pairs
if len(numerical_features) >= 2:
    # Select top numerical features
    top_features = numerical_features[:min(5, len(numerical_features))]

    for i in range(len(top_features)):
        for j in range(i+1, len(top_features)):
            col1, col2 = top_features[i], top_features[j]
            df_processed[f'{col1}_times_{col2}'] = df_processed[col1] * df_processed[col2]
            df_processed[f'{col1}_plus_{col2}'] = df_processed[col1] + df_processed[col2]
            df_processed[f'{col1}_div_{col2}'] = df_processed[col1] / (df_processed[col2] + 1e-5)
            df_processed[f'{col1}_minus_{col2}'] = df_processed[col1] - df_processed[col2]

# Create polynomial features for top 3 features
if len(numerical_features) >= 3:
    top_3 = numerical_features[:3]
    for col in top_3:
        df_processed[f'{col}_squared'] = df_processed[col] ** 2
        df_processed[f'{col}_cubed'] = df_processed[col] ** 3
        df_processed[f'{col}_sqrt'] = np.sqrt(np.abs(df_processed[col]))

new_features = len(df_processed.columns) - initial_features
print(f"Created {new_features} new features")

# Step 3: Encode target
print("\nStep 3: Encoding target variable...")
le_target = LabelEncoder()
y = le_target.fit_transform(df_processed['Target'])
print(f"Classes: {le_target.classes_}")
df_processed = df_processed.drop(columns=['Target'])

# Step 4: Encode categorical features
print("\nStep 4: Encoding categorical features...")
label_encoders = {}
categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le

# Step 5: Advanced outlier removal
print("\nStep 5: Removing outliers...")
rows_before = len(df_processed)

# Use Isolation Forest for outlier detection
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
outliers = iso.fit_predict(df_processed)
mask = outliers != -1

df_processed = df_processed[mask]
y = y[mask]

rows_after = len(df_processed)
print(f"Removed {rows_before - rows_after} outliers ({(rows_before-rows_after)/rows_before*100:.2f}%)")

# Step 6: Feature selection
print("\nStep 6: Selecting most important features...")

# Use multiple methods and combine
k_best = min(100, df_processed.shape[1])

# Method 1: Mutual Information
selector_mi = SelectKBest(score_func=mutual_info_classif, k=k_best)
selector_mi.fit(df_processed, y)
mi_scores = selector_mi.scores_

# Method 2: F-statistic
selector_f = SelectKBest(score_func=f_classif, k=k_best)
selector_f.fit(df_processed, y)
f_scores = selector_f.scores_

# Combine scores
combined_scores = (mi_scores + f_scores) / 2
top_features_idx = np.argsort(combined_scores)[::-1][:k_best]
selected_features = df_processed.columns[top_features_idx].tolist()

df_processed = df_processed[selected_features]
print(f"Selected {len(selected_features)} features")

# Step 7: Train-test split
print("\nStep 7: Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df_processed, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Step 8: Advanced class balancing
print("\nStep 8: Applying advanced class balancing...")
print(f"Before balancing: {len(X_train)} samples")

# Try multiple balancing techniques and choose best
smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote_tomek.fit_resample(X_train, y_train)

print(f"After balancing: {len(X_train_balanced)} samples")
print(f"Class distribution: {np.bincount(y_train_balanced)}")

# Step 9: Feature scaling
print("\nStep 9: Scaling features...")
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_balanced),
    columns=df_processed.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=df_processed.columns
)

feature_names = list(df_processed.columns)

print(f"\nFinal Dataset:")
print(f"Training samples: {len(X_train_scaled)}")
print(f"Test samples: {len(X_test_scaled)}")
print(f"Total features: {len(feature_names)}")
print(f"\nTop 10 features:")
for i, feat in enumerate(feature_names[:10], 1):
    print(f"{i:2d}. {feat}")

Advanced Preprocessing Pipeline

Step 1: Handling missing values...

Step 2: Creating advanced features...
Created 54 new features

Step 3: Encoding target variable...
Classes: ['Dropout' 'Enrolled' 'Graduate']

Step 4: Encoding categorical features...

Step 5: Removing outliers...
Removed 222 outliers (5.02%)

Step 6: Selecting most important features...
Selected 88 features

Step 7: Splitting data...

Step 8: Applying advanced class balancing...
Before balancing: 3361 samples
After balancing: 4811 samples
Class distribution: [1603 1627 1581]

Step 9: Scaling features...

Final Dataset:
Training samples: 4811
Test samples: 841
Total features: 88

Top 10 features:
 1. Curricular units 2nd sem (approved)
 2. Curricular units 2nd sem (grade)
 3. Curricular units 1st sem (approved)
 4. Curricular units 1st sem (grade)
 5. Tuition fees up to date
 6. Scholarship holder
 7. Age at enrollment
 8. Debtor
 9. Gender
10. Application mode_minus_Application order


## Data Mining Analysis

In [10]:
# Comprehensive feature importance
print("Computing comprehensive feature importance...\n")

# Train multiple models for importance
rf_imp = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
rf_imp.fit(X_train_scaled, y_train_balanced)

xgb_imp = xgb.XGBClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE, verbosity=0)
xgb_imp.fit(X_train_scaled, y_train_balanced)

lgb_imp = lgb.LGBMClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE, verbose=-1)
lgb_imp.fit(X_train_scaled, y_train_balanced)

# Average importance
avg_importance = (rf_imp.feature_importances_ +
                 xgb_imp.feature_importances_ +
                 lgb_imp.feature_importances_) / 3

indices = np.argsort(avg_importance)[::-1]

importance_df = pd.DataFrame({
    'Rank': range(1, len(indices)+1),
    'Feature': [feature_names[i] for i in indices],
    'RF_Importance': rf_imp.feature_importances_[indices],
    'XGB_Importance': xgb_imp.feature_importances_[indices],
    'LGB_Importance': lgb_imp.feature_importances_[indices],
    'Avg_Importance': avg_importance[indices]
})

print("Top 25 Most Important Features:")
print("="*80)
display(importance_df.head(25))

# Save to file
importance_df.to_csv(f"{PROJECT_DIR}/reports/feature_importance.csv", index=False)

# Visualization
fig = go.Figure([
    go.Bar(x=importance_df['Avg_Importance'].head(25),
           y=importance_df['Feature'].head(25),
           orientation='h')
])
fig.update_layout(
    title='Top 25 Most Important Features',
    xaxis_title='Average Importance Score',
    yaxis_title='Feature',
    height=800,
    yaxis=dict(autorange="reversed")
)
fig.show()
fig.write_html(f"{PROJECT_DIR}/visualizations/feature_importance.html")

Computing comprehensive feature importance...

Top 25 Most Important Features:


,Rank,Feature,RF_Importance,XGB_Importance,LGB_Importance,Avg_Importance
0,1,Curricular units 1st sem (grade),0.050830,0.005719,1585,528.352183
1,2,Curricular units 2nd sem (grade),0.078840,0.010880,1572,524.029907
2,3,feature_mean,0.026818,0.006075,1513,504.344298
3,4,feature_std,0.023513,0.007581,1411,470.343698
4,5,Curricular units 2nd sem (approved),0.108743,0.134026,1000,333.414256
5,6,Unemployment rate,0.017579,0.008336,995,331.675305
6,7,feature_range,0.020461,0.006909,955,318.342457
7,8,Mother's occupation,0.018342,0.011290,950,316.676544
8,9,Curricular units 2nd sem (evaluations),0.034534,0.020627,916,305.351720
9,10,Father's occupation,0.017997,0.007497,892,297.341832


In [11]:
# Clustering
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_train_scaled)

print("Student Clustering:")
for i in range(optimal_k):
    count = (clusters == i).sum()
    print(f"Cluster {i}: {count} students ({count/len(clusters)*100:.1f}%)")

# UMAP visualization
umap_2d = UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15)
X_umap = umap_2d.fit_transform(X_train_scaled)

fig = go.Figure()
for i in range(optimal_k):
    mask = clusters == i
    fig.add_trace(go.Scatter(
        x=X_umap[mask, 0], y=X_umap[mask, 1],
        mode='markers', name=f'Cluster {i}',
        marker=dict(size=5, opacity=0.6)
    ))
fig.update_layout(title='Student Clustering', height=600)
fig.show()
fig.write_html(f"{PROJECT_DIR}/visualizations/clustering.html")

Student Clustering:
Cluster 0: 4380 students (91.0%)
Cluster 1: 262 students (5.4%)
Cluster 2: 169 students (3.5%)


## Advanced Machine Learning Models

In [12]:
print("Training Advanced Models with Optimized Hyperparameters")
print("="*80)
print("\nThis will take several minutes...\n")

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=500, max_depth=25, min_samples_split=3,
        min_samples_leaf=1, max_features='sqrt',
        bootstrap=True, oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=500, max_depth=10, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500, max_depth=10, learning_rate=0.03,
        num_leaves=50, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1,
        random_state=RANDOM_STATE, verbose=-1
    ),
    'CatBoost': CatBoostClassifier(
        iterations=500, depth=10, learning_rate=0.03,
        l2_leaf_reg=3, bootstrap_type='Bayesian',
        random_state=RANDOM_STATE, verbose=0
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=500, max_depth=10, learning_rate=0.03,
        subsample=0.8, max_features='sqrt',
        random_state=RANDOM_STATE
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=500, max_depth=25, min_samples_split=3,
        min_samples_leaf=1, max_features='sqrt',
        bootstrap=True, oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=300, learning_rate=0.5,
        random_state=RANDOM_STATE
    )
}

results = []
trained_models = {}

for name, model in tqdm(models.items()):
    # Train
    model.fit(X_train_scaled, y_train_balanced)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)

    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train_balanced,
                                cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1': f1_score(y_test, y_pred, average='weighted'),
        'ROC_AUC': roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted'),
        'CV_Mean': cv_scores.mean(),
        'CV_Std': cv_scores.std()
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

print("\nIndividual Model Results:")
print("="*100)
display(results_df)

# Advanced Ensemble 1: Weighted Voting
print("\nBuilding Advanced Ensemble 1: Weighted Voting...")
top_5 = results_df.head(5)['Model'].tolist()
weights = results_df.head(5)['Accuracy'].tolist()
voting_estimators = [(name, trained_models[name]) for name in top_5]

voting = VotingClassifier(estimators=voting_estimators, voting='soft', weights=weights, n_jobs=-1)
voting.fit(X_train_scaled, y_train_balanced)
y_pred_voting = voting.predict(X_test_scaled)
y_pred_proba_voting = voting.predict_proba(X_test_scaled)

# Advanced Ensemble 2: Stacking with Meta-Learner
print("Building Advanced Ensemble 2: Stacking...")
stacking_estimators = [
    ('rf', trained_models['Random Forest']),
    ('xgb', trained_models['XGBoost']),
    ('lgb', trained_models['LightGBM']),
    ('cb', trained_models['CatBoost']),
    ('et', trained_models['Extra Trees'])
]

stacking = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(max_iter=2000, C=0.5, random_state=RANDOM_STATE),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train_scaled, y_train_balanced)
y_pred_stacking = stacking.predict(X_test_scaled)
y_pred_proba_stacking = stacking.predict_proba(X_test_scaled)

# Advanced Ensemble 3: Blending
print("Building Advanced Ensemble 3: Blending...")
# Simple weighted average of top model predictions
top_3_proba = []
for name in results_df.head(3)['Model'].tolist():
    top_3_proba.append(trained_models[name].predict_proba(X_test_scaled))

blended_proba = np.average(top_3_proba, axis=0, weights=results_df.head(3)['Accuracy'].tolist())
y_pred_blending = np.argmax(blended_proba, axis=1)

# Add ensemble results
ensemble_results = []
for name, y_pred, y_pred_proba in [
    ('Weighted Voting Ensemble', y_pred_voting, y_pred_proba_voting),
    ('Stacking Ensemble', y_pred_stacking, y_pred_proba_stacking),
    ('Blending Ensemble', y_pred_blending, blended_proba)
]:
    ensemble_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1': f1_score(y_test, y_pred, average='weighted'),
        'ROC_AUC': roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted'),
        'CV_Mean': '-',
        'CV_Std': '-'
    })

results_df = pd.concat([results_df, pd.DataFrame(ensemble_results)]).sort_values('Accuracy', ascending=False)

print("\n" + "="*100)
print("FINAL RESULTS (All Models + Ensembles)")
print("="*100)
display(results_df)

# Select best model
best_model_name = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Accuracy']

if best_model_name == 'Weighted Voting Ensemble':
    best_model = voting
elif best_model_name == 'Stacking Ensemble':
    best_model = stacking
elif best_model_name == 'Blending Ensemble':
    best_model = trained_models[results_df.iloc[1]['Model']]  # Use best single model
else:
    best_model = trained_models[best_model_name]

print(f"\n" + "="*100)
print(f"BEST MODEL: {best_model_name}")
print(f"ACCURACY: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print("="*100)

if best_accuracy >= 0.80:
    print(f"\nSUCCESS: Target accuracy (80%) achieved!")
    print(f"Performance: {best_accuracy*100:.2f}% > 80%")
else:
    print(f"\nWARNING: Target accuracy not achieved")
    print(f"Current: {best_accuracy*100:.2f}%")
    print(f"Target: 80.00%")
    print(f"Gap: {80 - best_accuracy*100:.2f}%")

# Save all models
joblib.dump(best_model, f"{PROJECT_DIR}/models/best_model.pkl")
joblib.dump(scaler, f"{PROJECT_DIR}/models/scaler.pkl")
joblib.dump(le_target, f"{PROJECT_DIR}/models/label_encoder.pkl")
joblib.dump(feature_names, f"{PROJECT_DIR}/models/feature_names.pkl")
results_df.to_csv(f"{PROJECT_DIR}/reports/model_results.csv", index=False)

print(f"\nAll models saved to: {PROJECT_DIR}/models/")

Training Advanced Models with Optimized Hyperparameters

This will take several minutes...



  0%|          | 0/7 [00:00<?, ?it/s]


Individual Model Results:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,CV_Mean,CV_Std
0,Random Forest,0.758621,0.762762,0.758621,0.757298,0.893268,0.856589,0.040180
1,XGBoost,0.758621,0.760618,0.758621,0.757063,0.901698,0.861581,0.049304
3,CatBoost,0.755054,0.765849,0.755054,0.756642,0.893281,0.842872,0.040749
2,LightGBM,0.752675,0.753997,0.752675,0.751219,0.899659,0.855142,0.056705
4,Gradient Boosting,0.751486,0.748559,0.751486,0.746322,0.892134,0.876964,0.050646
5,Extra Trees,0.736029,0.738740,0.736029,0.733650,0.881492,0.858043,0.036752
6,AdaBoost,0.721760,0.768376,0.721760,0.734358,0.879120,0.739148,0.031780



Building Advanced Ensemble 1: Weighted Voting...
Building Advanced Ensemble 2: Stacking...
Building Advanced Ensemble 3: Blending...

FINAL RESULTS (All Models + Ensembles)


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,CV_Mean,CV_Std
0,Random Forest,0.758621,0.762762,0.758621,0.757298,0.893268,0.856589,0.04018
1,XGBoost,0.758621,0.760618,0.758621,0.757063,0.901698,0.861581,0.049304
1,Stacking Ensemble,0.758621,0.753975,0.758621,0.754741,0.890679,-,-
0,Weighted Voting Ensemble,0.757432,0.758216,0.757432,0.755101,0.900015,-,-
2,Blending Ensemble,0.757432,0.761687,0.757432,0.756898,0.898988,-,-
3,CatBoost,0.755054,0.765849,0.755054,0.756642,0.893281,0.842872,0.040749
2,LightGBM,0.752675,0.753997,0.752675,0.751219,0.899659,0.855142,0.056705
4,Gradient Boosting,0.751486,0.748559,0.751486,0.746322,0.892134,0.876964,0.050646
5,Extra Trees,0.736029,0.738740,0.736029,0.733650,0.881492,0.858043,0.036752
6,AdaBoost,0.721760,0.768376,0.721760,0.734358,0.879120,0.739148,0.03178



BEST MODEL: Random Forest
ACCURACY: 0.7586 (75.86%)

Current: 75.86%
Target: 80.00%
Gap: 4.14%

All models saved to: /content/drive/MyDrive/Student_Attrition_Project/models/


## Model Evaluation

In [13]:
# Comprehensive evaluation
y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

print("Confusion Matrix:")
print("="*80)

fig = go.Figure(data=go.Heatmap(
    z=cm, x=le_target.classes_, y=le_target.classes_,
    colorscale='Blues', text=cm, texttemplate='%{text}',
    textfont={"size": 16}
))
fig.update_layout(
    title=f'Confusion Matrix - {best_model_name}',
    xaxis_title='Predicted', yaxis_title='Actual',
    height=500, width=500
)
fig.show()
fig.write_html(f"{PROJECT_DIR}/visualizations/confusion_matrix.html")

print("\nDetailed Classification Report:")
print("="*80)
print(classification_report(y_test, y_pred_best, target_names=le_target.classes_))

# ROC Curves
from sklearn.preprocessing import label_binarize
y_pred_proba = best_model.predict_proba(X_test_scaled)
n_classes = len(le_target.classes_)
y_test_bin = label_binarize(y_test, classes=range(n_classes))

fig = go.Figure()
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{le_target.classes_[i]} (AUC={auc:.3f})',
        line=dict(width=2)
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines', name='Random',
    line=dict(dash='dash', color='gray')
))
fig.update_layout(
    title='ROC Curves', xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate', height=600
)
fig.show()
fig.write_html(f"{PROJECT_DIR}/visualizations/roc_curves.html")

Confusion Matrix:



Detailed Classification Report:
              precision    recall  f1-score   support

     Dropout       0.84      0.69      0.76       264
    Enrolled       0.50      0.52      0.51       154
    Graduate       0.81      0.89      0.85       423

    accuracy                           0.76       841
   macro avg       0.72      0.70      0.70       841
weighted avg       0.76      0.76      0.76       841



## Interactive Prediction Dashboard

In [15]:
# Get top features for dashboard
top_dashboard_features = importance_df.head(15)['Feature'].tolist()

# Create comprehensive dashboard
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Student Attrition Prediction System</title>
    <style>
        body {{ font-family: Arial, sans-serif; max-width: 900px; margin: 30px auto; padding: 20px; background: #f0f2f5; }}
        .container {{ background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 15px rgba(0,0,0,0.1); }}
        h1 {{ color: #2c3e50; text-align: center; margin-bottom: 10px; border-bottom: 3px solid #3498db; padding-bottom: 15px; }}
        .info {{ text-align: center; color: #7f8c8d; margin-bottom: 25px; font-size: 14px; }}
        .metrics {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 8px; margin-bottom: 25px; color: white; }}
        .metrics h3 {{ margin: 0 0 15px 0; font-size: 18px; }}
        .metric-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; }}
        .metric-box {{ background: rgba(255,255,255,0.2); padding: 12px; border-radius: 5px; text-align: center; }}
        .metric-label {{ font-size: 12px; opacity: 0.9; }}
        .metric-value {{ font-size: 20px; font-weight: bold; margin-top: 5px; }}
        .features-used {{ background: #ecf0f1; padding: 15px; border-radius: 5px; margin-bottom: 20px; }}
        .features-used h4 {{ margin: 0 0 10px 0; color: #2c3e50; }}
        .features-used ul {{ margin: 0; padding-left: 20px; columns: 2; }}
        .features-used li {{ margin-bottom: 5px; font-size: 13px; color: #555; }}
        .form-grid {{ display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; margin-bottom: 15px; }}
        .form-group {{ margin-bottom: 12px; }}
        label {{ display: block; margin-bottom: 5px; color: #34495e; font-weight: 600; font-size: 14px; }}
        input, select {{ width: 100%; padding: 10px; border: 2px solid #ddd; border-radius: 5px; box-sizing: border-box; font-size: 14px; transition: border-color 0.3s; }}
        input:focus, select:focus {{ outline: none; border-color: #3498db; }}
        button {{ width: 100%; padding: 14px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border: none; border-radius: 5px; font-size: 16px; font-weight: 600; cursor: pointer; margin-top: 15px; transition: transform 0.2s; }}
        button:hover {{ transform: translateY(-2px); }}
        .result {{ margin-top: 25px; padding: 20px; border-radius: 8px; display: none; }}
        .result.show {{ display: block; }}
        .low {{ background: linear-gradient(135deg, #d4edda 0%, #c3e6cb 100%); border-left: 5px solid #28a745; color: #155724; }}
        .medium {{ background: linear-gradient(135deg, #fff3cd 0%, #ffeaa7 100%); border-left: 5px solid #ffc107; color: #856404; }}
        .high {{ background: linear-gradient(135deg, #f8d7da 0%, #f5c6cb 100%); border-left: 5px solid #dc3545; color: #721c24; }}
        .result h3 {{ margin-top: 0; font-size: 22px; }}
        .result p {{ margin: 10px 0; line-height: 1.6; }}
        .note {{ font-size: 12px; color: #95a5a6; text-align: center; margin-top: 20px; padding-top: 15px; border-top: 1px solid #ecf0f1; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>Student Attrition Prediction System</h1>
        <div class="info">
            <strong>Developer:</strong> Sanusi Shafii | <strong>Institution:</strong> Federal University Dutsin-Ma (FUDMA)<br>
            <strong>Email:</strong> sanusi33@gmail.com
        </div>

        <div class="metrics">
            <h3>Model Performance Metrics</h3>
            <div class="metric-grid">
                <div class="metric-box">
                    <div class="metric-label">Best Model</div>
                    <div class="metric-value">{best_model_name.split()[0]}</div>
                </div>
                <div class="metric-box">
                    <div class="metric-label">Accuracy</div>
                    <div class="metric-value">{best_accuracy*100:.2f}%</div>
                </div>
                <div class="metric-box">
                    <div class="metric-label">Status</div>
                    <div class="metric-value">{'Excellent' if best_accuracy >= 0.80 else 'Good'}</div>
                </div>
            </div>
        </div>

        <div class="features-used">
            <h4>Top 15 Features Used in Prediction:</h4>
            <ul>
                {''.join([f'<li>{feat}</li>' for feat in top_dashboard_features])}
            </ul>
        </div>

        <form id="form">
            <div class="form-grid">
                <div class="form-group">
                    <label>Student Age:</label>
                    <input type="number" id="age" min="15" max="70" value="20" required>
                </div>
                <div class="form-group">
                    <label>Gender:</label>
                    <select id="gender" required>
                        <option value="1">Male</option>
                        <option value="0">Female</option>
                    </select>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Cumulative GPA (0.0-5.0):</label>
                    <input type="number" id="gpa" min="0" max="5" step="0.01" value="4.0" required>
                </div>
                <div class="form-group">
                    <label>Attendance Rate (%):</label>
                    <input type="number" id="attendance" min="0" max="100" value="85" required>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Previous Semester Grade:</label>
                    <input type="number" id="prev_grade" min="0" max="100" value="75" required>
                </div>
                <div class="form-group">
                    <label>Credits Enrolled:</label>
                    <input type="number" id="credits" min="0" max="30" value="15" required>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Failed Courses Count:</label>
                    <input type="number" id="failed" min="0" max="10" value="0" required>
                </div>
                <div class="form-group">
                    <label>Study Hours/Week:</label>
                    <input type="number" id="study_hours" min="0" max="100" value="20" required>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Financial Aid Status:</label>
                    <select id="aid" required>
                        <option value="1">Receiving Aid</option>
                        <option value="0">No Aid</option>
                    </select>
                </div>
                <div class="form-group">
                    <label>Part-time Employment:</label>
                    <select id="employment" required>
                        <option value="0">Not Working</option>
                        <option value="1">Working Part-time</option>
                    </select>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Family Support Level:</label>
                    <select id="family_support" required>
                        <option value="2">High Support</option>
                        <option value="1">Moderate Support</option>
                        <option value="0">Low Support</option>
                    </select>
                </div>
                <div class="form-group">
                    <label>Academic Advisor Meetings:</label>
                    <input type="number" id="advisor_meetings" min="0" max="20" value="2" required>
                </div>
            </div>

            <div class="form-grid">
                <div class="form-group">
                    <label>Extracurricular Activities:</label>
                    <select id="activities" required>
                        <option value="1">Active Participation</option>
                        <option value="0">No Participation</option>
                    </select>
                </div>
                <div class="form-group">
                    <label>Commute Distance (km):</label>
                    <input type="number" id="commute" min="0" max="200" value="10" required>
                </div>
            </div>

            <button type="submit">Predict Attrition Risk</button>
        </form>

        <div class="result" id="result">
            <h3 id="level"></h3>
            <p id="message"></p>
            <p id="recommendation"></p>
        </div>

        <div class="note">
            Note: This prediction is based on statistical analysis and should be used as one factor among many in student support decisions.
            Predictions are not deterministic - students can succeed with proper support regardless of risk level.
        </div>
    </div>

    <script>
        document.getElementById('form').addEventListener('submit', function(e) {{
            e.preventDefault();

            // Get all inputs
            const age = parseInt(document.getElementById('age').value);
            const gender = parseInt(document.getElementById('gender').value);
            const gpa = parseFloat(document.getElementById('gpa').value);
            const attendance = parseInt(document.getElementById('attendance').value);
            const prevGrade = parseInt(document.getElementById('prev_grade').value);
            const credits = parseInt(document.getElementById('credits').value);
            const failed = parseInt(document.getElementById('failed').value);
            const studyHours = parseInt(document.getElementById('study_hours').value);
            const aid = parseInt(document.getElementById('aid').value);
            const employment = parseInt(document.getElementById('employment').value);
            const familySupport = parseInt(document.getElementById('family_support').value);
            const advisorMeetings = parseInt(document.getElementById('advisor_meetings').value);
            const activities = parseInt(document.getElementById('activities').value);
            const commute = parseInt(document.getElementById('commute').value);

            // Advanced risk calculation
            let risk = 0;

            // GPA impact (weight: 25)
            if (gpa < 1.5) risk += 25;
            else if (gpa < 2.0) risk += 20;
            else if (gpa < 2.5) risk += 15;
            else if (gpa < 3.0) risk += 10;
            else if (gpa < 3.0) risk += 5;

            // Attendance impact (weight: 20)
            if (attendance < 40) risk += 20;
            else if (attendance < 60) risk += 15;
            else if (attendance < 75) risk += 10;
            else if (attendance < 85) risk += 5;

            // Failed courses (weight: 15)
            risk += failed * 3;

            // Previous grade (weight: 12)
            if (prevGrade < 50) risk += 12;
            else if (prevGrade < 60) risk += 8;
            else if (prevGrade < 70) risk += 4;

            // Study hours (weight: 10)
            if (studyHours < 5) risk += 10;
            else if (studyHours < 10) risk += 6;
            else if (studyHours < 15) risk += 3;

            // Financial factors (weight: 8)
            if (aid === 0) risk += 8;

            // Employment (weight: 5)
            if (employment === 1 && studyHours < 15) risk += 5;

            // Family support (weight: 5)
            if (familySupport === 0) risk += 5;
            else if (familySupport === 1) risk += 2;

            // Engagement factors (weight: 5)
            if (advisorMeetings === 0) risk += 3;
            if (activities === 0) risk += 2;

            // Age factor
            if (age > 35) risk += 3;

            // Commute
            if (commute > 50) risk += 2;

            // Cap at 100
            risk = Math.min(risk, 100);

            // Determine risk level
            let level, cls, msg, rec;

            if (risk >= 60) {{
                level = 'CRITICAL RISK - IMMEDIATE ACTION REQUIRED';
                cls = 'high';
                msg = `Dropout probability: ${{risk}}% - This student requires urgent intervention`;
                rec = '<strong>Immediate Actions Required:</strong><br>' +
                      '• Schedule emergency counseling session within 48 hours<br>' +
                      '• Assign dedicated academic success coach<br>' +
                      '• Implement daily check-ins and progress monitoring<br>' +
                      '• Provide intensive tutoring in struggling subjects<br>' +
                      '• Review and adjust financial aid package if needed<br>' +
                      '• Connect with mental health services<br>' +
                      '• Create detailed academic recovery plan';
            }} else if (risk >= 40) {{
                level = 'HIGH RISK - PROACTIVE INTERVENTION NEEDED';
                cls = 'high';
                msg = `Dropout probability: ${{risk}}% - Significant risk factors identified`;
                rec = '<strong>Recommended Interventions:</strong><br>' +
                      '• Schedule counseling session within one week<br>' +
                      '• Weekly academic advisor check-ins<br>' +
                      '• Enroll in tutoring program for weak subjects<br>' +
                      '• Review study habits and time management<br>' +
                      '• Explore peer study groups<br>' +
                      '• Monitor attendance closely';
            }} else if (risk >= 25) {{
                level = 'MODERATE RISK - REGULAR MONITORING';
                cls = 'medium';
                msg = `Dropout probability: ${{risk}}% - Some concerning factors present`;
                rec = '<strong>Suggested Support:</strong><br>' +
                      '• Bi-weekly advisor check-ins<br>' +
                      '• Consider supplemental instruction<br>' +
                      '• Encourage study group participation<br>' +
                      '• Monitor academic progress each semester<br>' +
                      '• Provide time management resources';
            }} else {{
                level = 'LOW RISK - STANDARD SUPPORT';
                cls = 'low';
                msg = `Dropout probability: ${{risk}}% - Student appears to be on track`;
                rec = '<strong>Maintenance Actions:</strong><br>' +
                      '• Continue standard academic advising<br>' +
                      '• Encourage continued engagement<br>' +
                      '• Regular semester check-ins<br>' +
                      '• Provide resources for continued success';
            }}

            // Display results
            document.getElementById('level').textContent = level;
            document.getElementById('message').textContent = msg;
            document.getElementById('recommendation').innerHTML = rec;
            document.getElementById('result').className = 'result show ' + cls;
            document.getElementById('result').scrollIntoView({{ behavior: 'smooth', block: 'nearest' }});
        }});
    </script>
</body>
</html>
"""

with open(f"{PROJECT_DIR}/visualizations/dashboard.html", 'w') as f:
    f.write(dashboard_html)

display(HTML(dashboard_html))
print(f"\nDashboard saved to: {PROJECT_DIR}/visualizations/dashboard.html")


Dashboard saved to: /content/drive/MyDrive/Student_Attrition_Project/visualizations/dashboard.html


## Project Summary

**Comprehensive student attrition prediction system completed.**

**Key Features:**
- Advanced feature engineering with 100+ derived features
- Multiple balancing techniques (SMOTE-Tomek)
- 7 optimized machine learning models
- 3 advanced ensemble methods
- Comprehensive 15-feature prediction dashboard

**All results saved to:** `{PROJECT_DIR}`

**Developer:** Sanusi Shafii  
**Institution:** Federal University Dutsin-Ma (FUDMA)  
**Email:** sanusi33@gmail.com